# Central Bank of Sri Lanka — exchange rates

**Status: planned. There is no connector, and no data.** Owner: M1 Dinapura.

This notebook is not a scaffold waiting to be filled in. It documents a gap
that is currently real, shows the evidence for it, and records what is known
about the source so whoever builds the connector does not rediscover it.

CBSL is the authority for USD/LKR. CeyNex needs it for one job: converting
between USD and rupees when a question asks about a currency shock.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. The gap, shown rather than asserted

`fact_trade` already has an `fx_usd_lkr` column — the contract made room for
exchange rates from the start. Nothing fills it.

In [2]:
facts = cx.comtrade_parquet()
total = len(facts)
filled = facts["fx_usd_lkr"].notna().sum()

print(f"fact_trade rows on disk : {total:,}")
print(f"rows with fx_usd_lkr    : {filled:,}")
print(f"rows without            : {total - filled:,}  ({100 * (total - filled) / total:.1f}%)")
print()
print("The column exists in the frozen contract and is entirely empty.")
print("That is the CBSL connector's absence, visible in the data.")

fact_trade rows on disk : 4,625
rows with fx_usd_lkr    : 0
rows without            : 4,625  (100.0%)

The column exists in the frozen contract and is entirely empty.
That is the CBSL connector's absence, visible in the data.


## 2. What runs instead today

Currency-shock questions do not fail. The Trade Economics agent uses a
documented elasticity from `config/elasticities.yaml` and states that
assumption in every answer that relies on it.

So the missing connector costs precision, not capability — and the system says
so out loud rather than implying a number it does not have.

In [3]:
def yaml_block(text, key):
    """Return the top-level `key:` block from a YAML file, comments and all."""
    out, inside = [], False
    for line in text.splitlines():
        if line.startswith(f"{key}:"):
            inside = True
        elif inside and line and not line[0].isspace() and not line.startswith("#"):
            break
        if inside:
            out.append(line)
    return "\n".join(out)


elasticities = cx.CONFIG / "elasticities.yaml"
if elasticities.exists():
    print(f"{elasticities}\n")
    print(yaml_block(elasticities.read_text(), "fx_pass_through"))
else:
    print(f"not found: {elasticities}")

/ml/CeyNex/ceynex-core/config/elasticities.yaml

fx_pass_through:
  # Share of a depreciation that reaches USD export prices within one year.
  # <1.0 because contracts are priced in USD and repriced with a lag.
  agriculture:
    value: 0.6
    basis: literature_range
    source: "TBD Day 7 — cite the source used"
  apparel:
    value: 0.4
    basis: literature_range
    source: "TBD Day 7 — apparel is more contract-priced than agriculture"



## 3. What the connector will have to handle

One thing is already known and is the kind of detail that silently corrupts a
series if it is missed.

> **The middle-rate USD/LKR series was discontinued on 07.03.2023.**
> After that date the equivalent series is the **indicative spot rate**.

They are different series. Joining them end to end without noting the switch
produces one continuous-looking rate history with a methodology change hidden
inside it, which is exactly the sort of thing a forecast will happily fit and
nobody will question.

**Sources when it gets built:**

| What | Where |
|---|---|
| Daily and monthly USD/LKR | CBSL Economic Data Library |
| Rate methodology change | CBSL rate notices, 07.03.2023 |

**Where it lands:** the `fx_usd_lkr` column measured empty above, plus its own
`dq_flag` on any row spanning the March 2023 methodology change.

## 4. What not to do

Do not backfill `fx_usd_lkr` from a third-party FX API to make the column look
populated. Every other number in `fact_trade` traces to a named source with a
`source_hash`; an FX rate from a convenience API would be the one value in the
table nobody could check against a primary source. An empty column is honest.
An unattributed one is not.